# DATA PREPARATION - Movie Recommendation System

## 1. Import Neccessary Libraries

In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import time
import os
import sys
from tqdm import tqdm 
import warnings
warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))
from api_key import TMDB_API_KEY

movies = pd.read_csv('../data/ml-latest-small/movies.csv')
ratings = pd.read_csv('../data/ml-latest-small/ratings.csv')
links = pd.read_csv('../data/ml-latest-small/links.csv')
tags = pd.read_csv('../data/ml-latest-small/tags.csv')

print(f"Movies shape: {movies.shape}")
print(f"Ratings shape: {ratings.shape}")
print(f"Links shape: {links.shape}")
print(f"Tags shape: {tags.shape}")

Movies shape: (9742, 3)
Ratings shape: (100836, 4)
Links shape: (9742, 3)
Tags shape: (3683, 4)


In [29]:
# lookup in the datasets
print("\nSample from movies:")
movies.sample(3)


Sample from movies:


,movieId,title,genres
5590,26870,Major League II (1994),Comedy
2035,2710,"Blair Witch Project, The (1999)",Drama|Horror|Thriller
4551,6763,Duplex (2003),Comedy|Crime


In [30]:
print("\nSample from links:")
links.sample(3)


Sample from links:


,movieId,imdbId,tmdbId
3123,4202,68617,4986.0
5953,34397,193364,37632.0
7780,91688,1251743,51995.0


In [31]:
print("\nSample from ratings:")
ratings.sample(3)


Sample from ratings:


,userId,movieId,rating,timestamp
41306,280,2724,3.0,1348532965
64831,414,87304,4.0,1485657233
22235,149,737,2.0,902085168


In [32]:
print("\nSample from tags:")
tags.sample(3)


Sample from tags:


,userId,movieId,tag,timestamp
1001,474,32,time travel,1137206826
3495,599,296,nonlinear,1498456341
3677,606,6107,World War II,1178473747


## 2. Merge Datasets

In [ ]:
# tmdbId to fetch rich metadata from TMDB
print("\nMerging movies with links on movieId...")

# First, merging movies and links
movie_links = pd.merge(movies, links, on='movieId', how='left')

# Keeping only useful columns for now
movie_links = movie_links[['movieId', 'title', 'genres', 'tmdbId']]

print(f"Merged dataset shape: {movie_links.shape}")
print("Sample after merge:")
movie_links.head()


Merging movies with links on movieId...
Merged dataset shape: (9742, 4)
Sample after merge:


,movieId,title,genres,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,862.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,8844.0
2,3,Grumpier Old Men (1995),Comedy|Romance,15602.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,31357.0
4,5,Father of the Bride Part II (1995),Comedy,11862.0


In [36]:
if not TMDB_API_KEY:
    raise ValueError("TMDB API key missing!")

In [38]:
# Base URL for TMDB movie details
BASE_URL = "https://api.themoviedb.org/3/movie/"

# Headers 
headers = {
    "accept": "application/json",
    "Authorization": f"Bearer {TMDB_API_KEY}"   
}

## 3. CACHE SETUP

In [ ]:
# Using simple JSON file to cache TMDB responses so we don't hit rate limits
# or re-download the same movie multiple times.

cache_file = '../cache/tmdb_cache.json'

if os.path.exists(cache_file):
    with open(cache_file, 'r', encoding='utf-8') as f:
        tmdb_cache = json.load(f)
    print(f"Loaded existing cache with {len(tmdb_cache)} movies from ../cache/tmdb_cache.json")
else:
    tmdb_cache = {}
    print(" No existing cache found. Starting with an empty cache.")

 No existing cache found. Starting with an empty cache.


## 4. TMDB Fetch Functions 

In [43]:
def get_tmdb_metadata(tmdb_id):
    """Fetch movie metadata from TMDB with caching."""
    if tmdb_id is None or pd.isna(tmdb_id):
        return None
    
    tmdb_id = int(tmdb_id)
    str_id = str(tmdb_id)
    
    # Return cached result if available
    if str_id in tmdb_cache:
        return tmdb_cache[str_id]
    
    try:
        url = f"https://api.themoviedb.org/3/movie/{tmdb_id}?language=en-US"
        
        # Using API key in query params 
        params = {'api_key': TMDB_API_KEY}
        
        response = requests.get(url, params=params, timeout=12)
        
        if response.status_code == 200:
            data = response.json()
            metadata = {
                'overview': data.get('overview', ''),
                'poster_path': data.get('poster_path'),
                'release_date': data.get('release_date', ''),
                'vote_average': data.get('vote_average'),
                'tmdb_genres': [g['name'] for g in data.get('genres', [])]
            }
            tmdb_cache[str_id] = metadata
            time.sleep(0.22)          
            return metadata
            
        elif response.status_code == 404:
            # Movie not found on TMDB
            tmdb_cache[str_id] = None
            return None
        else:
            print(f" TMDB API error {response.status_code} for tmdbId {tmdb_id}")
            tmdb_cache[str_id] = None
            time.sleep(1)
            return None
            
    except Exception as e:
        print(f" Exception for tmdbId {tmdb_id}: {e}")
        tmdb_cache[str_id] = None
        time.sleep(1)
        return None

## 5. ENRICH MOVIES - with progress bar

In [44]:
print("\nEnriching movies with TMDB metadata.")

enriched_rows = []

for idx, row in tqdm(movie_links.iterrows(), total=len(movie_links), desc="Fetching TMDB data"):
    meta = get_tmdb_metadata(row['tmdbId'])
    
    if meta:
        enriched_rows.append({
            'movieId': row['movieId'],
            'title': row['title'],
            'genres': row['genres'],                   
            'tmdbId': row['tmdbId'],
            'overview': meta['overview'],
            'poster_path': meta['poster_path'],
            'release_date': meta['release_date'],
            'vote_average': meta['vote_average'],
            'tmdb_genres': meta['tmdb_genres']
        })
    else:
        # Fallback
        enriched_rows.append({
            'movieId': row['movieId'],
            'title': row['title'],
            'genres': row['genres'],
            'tmdbId': row['tmdbId'],
            'overview': '',
            'poster_path': None,
            'release_date': '',
            'vote_average': None,
            'tmdb_genres': []
        })

enriched_movies = pd.DataFrame(enriched_rows)


Enriching movies with TMDB metadata.


Fetching TMDB data: 100%|███████████████████| 9742/9742 [59:43<00:00,  2.72it/s]


## 6.  SAVING  

In [46]:
print("\n6. Saving enriched data to processed folder...")

processed_dir = '../data/processed'

enriched_path = os.path.join(processed_dir, 'enriched_movies.csv')
enriched_movies.to_csv(enriched_path, index=False)

# Save a sample for quick testing
sample_path = os.path.join(processed_dir, 'enriched_movies_sample.csv')
enriched_movies.head(2000).to_csv(sample_path, index=False)

print(f" Saved full dataset : {enriched_path}")
print(f" Saved sample       : {sample_path}")
print(f" Total movies saved    : {len(enriched_movies)}")



6. Saving enriched data to processed folder...
 Saved full dataset : ../data/processed/enriched_movies.csv
 Saved sample       : ../data/processed/enriched_movies_sample.csv
 Total movies saved    : 9742


## 7. CREATE CONTENT TEXT FOR LATER EMBEDDINGS 

In [47]:
print("\n7. Creating 'content_text' column for semantic search...")

def create_content_text(row):
    """Build rich text by combining title, genres, and overview."""
    parts = []
    
    if pd.notna(row.get('title')) and row['title']:
        parts.append(str(row['title']).strip())
    
    if pd.notna(row.get('genres')) and row['genres'] != '(no genres listed)':
        genres_clean = row['genres'].replace('|', ', ')
        parts.append(genres_clean)
    
    if pd.notna(row.get('overview')) and row['overview']:
        parts.append(str(row['overview']).strip())
    
    if isinstance(row.get('tmdb_genres'), list) and row['tmdb_genres']:
        parts.append(', '.join(row['tmdb_genres']))
    
    return " | ".join([p for p in parts if p])

enriched_movies['content_text'] = enriched_movies.apply(create_content_text, axis=1)

# Save final version with content_text
enriched_movies.to_csv(enriched_path, index=False)
print(" Added 'content_text' column and updated file.")


7. Creating 'content_text' column for semantic search...
 Added 'content_text' column and updated file.


## 8. SAVE UPDATED CACHE

In [48]:
print("\n8. Saving TMDB cache...")
with open(cache_file, 'w', encoding='utf-8') as f:
    json.dump(tmdb_cache, f, ensure_ascii=False, indent=2)

print(f" Cache saved to: {cache_file} ({len(tmdb_cache)} entries)")

print("📁 Files created/updated:")
print(f"   • ../data/processed/enriched_movies.csv")
print(f"   • ../data/processed/enriched_movies_sample.csv")
print(f"   • ../cache/tmdb_cache.json")
print(f"\nTotal enriched movies: {len(enriched_movies)}")


8. Saving TMDB cache...
 Cache saved to: ../cache/tmdb_cache.json (9733 entries)
📁 Files created/updated:
   • ../data/processed/enriched_movies.csv
   • ../data/processed/enriched_movies_sample.csv
   • ../cache/tmdb_cache.json

Total enriched movies: 9742
